# 05 - BiLSTM y BiGRU (Bidireccional)
## Clasificación de Marcha Patológica

**Autor:** Weimar Andres Arenas Gonzalez  
**Curso:** Fundamentos de Deep Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WeimarArenas/ProyectoDL20261/blob/main/05%20-%20BiLSTM_BiGRU.ipynb)

---
**Objetivo:** Explorar si las redes bidireccionales mejoran la clasificación al capturar
dependencias tanto hacia adelante como hacia atrás en la secuencia de marcha.

**Motivación:** La marcha tiene patrones cíclicos; una red bidireccional puede detectar
anticipaciones de movimiento (p.ej., levantamiento del pie antes de posarse) que una
red unidireccional no ve hasta que ocurre.

**Referencia externa:** Bi-LSTM en dataset similar → 90.75% (Guo et al., 2020)

## 1. Configuración del entorno

In [ ]:
# ============================================================
# FUNCIONES DE CARGA — Formato real del dataset GIST
# CSV: tab-separado, sin header, 101 columnas
# Col 0: timestamp | cols (1+j*4)+1,+2,+3 = x,y,z para joint j
# Directorio padre: human{N}_{gait_type}{rep}/
# ============================================================

import re as _re

# Indices de features: x,y,z de cada articulacion (omite timestamp y joint_id)
FEATURE_COLS = []
for _j in range(25):
    _base = 1 + _j * 4
    FEATURE_COLS.extend([_base + 1, _base + 2, _base + 3])
# Resultado: [2,3,4, 6,7,8, ..., 98,99,100]  -> 75 valores por fila

_GAIT_PAT = _re.compile(
    r'human(\d+)_(normal|antalgic|lurch|steppage|stiff_legged|trendelenburg)\d+'
)
CLASS_MAP = {
    'normal': 0, 'antalgic': 1, 'stiff_legged': 2,
    'lurch': 3, 'steppage': 4, 'trendelenburg': 5,
}
CLASS_NAMES = ['Normal', 'Antalgic', 'Stiff-legged', 'Lurching', 'Steppage', 'Trendelenburg']


def extract_label_and_subject(filepath):
    """Extrae clase y sujeto del nombre del directorio padre del CSV."""
    dirname = os.path.basename(os.path.dirname(filepath))
    m = _GAIT_PAT.match(dirname)
    if not m:
        return None, None
    return CLASS_MAP.get(m.group(2)), int(m.group(1))


def load_csv_sequence(filepath, skip_frames=10, n_frames=50):
    """
    Lee un CSV del GIST dataset (tab-separado, 101 cols, sin header).
    Retorna array (n_frames, 75) omitiendo los primeros skip_frames.
    """
    try:
        df = pd.read_csv(filepath, header=None, sep='\t')
        if df.shape[1] != 101 or len(df) < skip_frames + n_frames:
            return None
        seq = df.iloc[skip_frames:skip_frames + n_frames, FEATURE_COLS].values.astype(np.float32)
        return None if np.any(np.isnan(seq)) else seq
    except Exception:
        return None


def load_full_dataset(data_dir, verbose=True):
    """
    Carga todos los CSV validos del GIST dataset.
    data_dir: ruta a Pathological_Gaits/
    Retorna: X (N,50,75), y (N,), S (N,)
    """
    sequences, labels, subjects = [], [], []
    skipped = 0
    for dirname in sorted(os.listdir(data_dir)):
        dirpath = os.path.join(data_dir, dirname)
        if not os.path.isdir(dirpath):
            continue
        for fname in sorted(os.listdir(dirpath)):
            if not fname.endswith('.csv'):
                continue
            fpath = os.path.join(dirpath, fname)
            label, subject = extract_label_and_subject(fpath)
            if label is None:
                skipped += 1
                continue
            seq = load_csv_sequence(fpath)
            if seq is None:
                skipped += 1
                continue
            sequences.append(seq)
            labels.append(label)
            subjects.append(subject)
    if verbose:
        print(f'Secuencias cargadas: {len(sequences)} | Descartadas: {skipped}')
    return (np.array(sequences, dtype=np.float32),
            np.array(labels, dtype=np.int32),
            np.array(subjects, dtype=np.int32))


In [ ]:
# ---- Cargar datos (desde .npy cache o desde CSV) ----------------------------
if IN_COLAB and not os.path.exists('pathological_gait_datasets'):
    os.system('git clone --quiet https://github.com/kooksung/pathological_gait_datasets.git')

DATA_DIR = 'pathological_gait_datasets/Pathological_Gaits'

if os.path.exists('X_all.npy') and os.path.exists('y_labels.npy'):
    X    = np.load('X_all.npy')
    y    = np.load('y_labels.npy')
    S    = np.load('S_subjects.npy')
    print(f'Datos cargados desde .npy: X={X.shape}')
else:
    assert os.path.isdir(DATA_DIR), f'No se encontro el dataset en: {DATA_DIR}'
    X, y, S = load_full_dataset(DATA_DIR)
    np.save('X_all.npy', X)
    np.save('y_labels.npy', y)
    np.save('S_subjects.npy', S)
    print(f'Datos cargados desde CSV y guardados: X={X.shape}')

# Seleccion de joints de piernas
LEG_JOINT_INDICES = [0, 1, 12, 13, 14, 15, 16, 17, 18, 19]
leg_feat = [f for j in LEG_JOINT_INDICES for f in [j*3, j*3+1, j*3+2]]
X_legs = X[:, :, leg_feat]

print(f'X={X.shape} | X_legs={X_legs.shape}')
print(f'Clases: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'Sujetos: {sorted(np.unique(S))}')


## 2. Arquitecturas Bidireccionales

Las redes bidireccionales procesan la secuencia tanto en dirección adelante (t=0→49)
como hacia atrás (t=49→0), concatenando los estados ocultos de ambas direcciones.
Esto duplica el tamaño de salida efectivo: n_units × 2.

In [ ]:
def build_bilstm(input_shape, n_classes=6, n_layers=2, n_units=64,
                 dropout_rate=0.3, l2_reg=1e-4):
    """
    BiLSTM de n_layers capas bidireccionales.
    Cada capa Bidireccional duplica las unidades efectivas: n_units → 2*n_units.
    Se reduce n_units respecto a la versión unidireccional para mantener
    un número de parámetros comparable.
    """
    reg = keras.regularizers.l2(l2_reg)
    
    inputs = layers.Input(shape=input_shape, name='input')
    x = layers.Dense(n_units * 2, activation='relu', kernel_regularizer=reg, name='proj')(inputs)
    
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        lstm_layer = layers.LSTM(
            n_units,
            return_sequences=return_seq,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            name=f'lstm_{i+1}'
        )
        # Bidireccional: procesa la secuencia en ambas direcciones
        x = layers.Bidirectional(lstm_layer, merge_mode='concat', name=f'bilstm_{i+1}')(x)
        if return_seq:
            x = layers.Dropout(dropout_rate, name=f'drop_{i+1}')(x)
    
    x = layers.Dense(n_units * 2, activation='relu', kernel_regularizer=reg, name='fc')(x)
    x = layers.Dropout(dropout_rate, name='drop_fc')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='softmax')(x)
    
    return keras.Model(inputs, outputs, name='BiLSTM_Classifier')


def build_bigru(input_shape, n_classes=6, n_layers=2, n_units=64,
                dropout_rate=0.3, l2_reg=1e-4):
    """
    BiGRU de n_layers capas bidireccionales.
    GRU tiene menos parámetros que LSTM (no hay forget gate separado),
    lo que puede dar ventaja en datasets no muy grandes.
    """
    reg = keras.regularizers.l2(l2_reg)
    
    inputs = layers.Input(shape=input_shape, name='input')
    x = layers.Dense(n_units * 2, activation='relu', kernel_regularizer=reg, name='proj')(inputs)
    
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        gru_layer = layers.GRU(
            n_units,
            return_sequences=return_seq,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            name=f'gru_{i+1}'
        )
        x = layers.Bidirectional(gru_layer, merge_mode='concat', name=f'bigru_{i+1}')(x)
        if return_seq:
            x = layers.Dropout(dropout_rate, name=f'drop_{i+1}')(x)
    
    x = layers.Dense(n_units * 2, activation='relu', kernel_regularizer=reg, name='fc')(x)
    x = layers.Dropout(dropout_rate, name='drop_fc')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='softmax')(x)
    
    return keras.Model(inputs, outputs, name='BiGRU_Classifier')


# Comparar parámetros
print('=== Comparación de parámetros ===')
for name, builder, shape in [
    ('BiLSTM (75 feat)', build_bilstm, (50, 75)),
    ('BiGRU  (75 feat)', build_bigru,  (50, 75)),
    ('BiLSTM (30 feat)', build_bilstm, (50, 30)),
    ('BiGRU  (30 feat)', build_bigru,  (50, 30)),
]:
    m = builder(shape)
    print(f'  {name}: {m.count_params():,} parámetros')

# Mostrar resumen del BiGRU
print('\n=== Arquitectura BiGRU (75 feat) ===')
build_bigru((50, 75)).summary()

## 3. Funciones de entrenamiento y evaluación LOSO-CV

In [ ]:
def evaluate_fold(model, X_test, y_test):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    sensitivity = np.zeros(N_CLASSES)
    specificity = np.zeros(N_CLASSES)
    precision = np.zeros(N_CLASSES)
    for cls in range(N_CLASSES):
        TP = cm[cls, cls]; FN = cm[cls, :].sum() - TP
        FP = cm[:, cls].sum() - TP; TN = cm.sum() - TP - FN - FP
        sensitivity[cls] = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity[cls] = TN / (TN + FP) if (TN + FP) > 0 else 0
        precision[cls]   = TP / (TP + FP) if (TP + FP) > 0 else 0
    return {'accuracy': acc, 'confusion_matrix': cm,
            'sensitivity': sensitivity, 'specificity': specificity, 'precision': precision}


def train_loso_cv(X_data, y, S, model_builder_fn, model_name,
                  n_epochs=500, batch_size=64, patience=30):
    logo = LeaveOneGroupOut()
    results = []
    print(f'\n=== LOSO-CV: {model_name} ===')
    
    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_data, y, groups=S)):
        test_subj = int(np.unique(S[test_idx])[0])
        N_tr, T, F = X_data[train_idx].shape
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_data[train_idx].reshape(-1, F)).reshape(N_tr, T, F)
        X_te = scaler.transform(X_data[test_idx].reshape(-1, F)).reshape(len(test_idx), T, F)
        y_tr_oh = to_categorical(y[train_idx], num_classes=N_CLASSES)
        
        model = model_builder_fn((T, F))
        model.compile(optimizer=keras.optimizers.Adam(1e-3),
                      loss='categorical_crossentropy', metrics=['accuracy'])
        
        t0 = time.time()
        hist = model.fit(X_tr, y_tr_oh, epochs=n_epochs, batch_size=batch_size,
                         validation_split=0.1,
                         callbacks=[
                             EarlyStopping('val_accuracy', patience=patience,
                                           restore_best_weights=True, verbose=0),
                             ReduceLROnPlateau('val_loss', factor=0.5, patience=15,
                                               min_lr=1e-6, verbose=0)
                         ], verbose=0)
        
        metrics = evaluate_fold(model, X_te, y[test_idx])
        metrics.update({'fold': fold_idx, 'test_subject': test_subj,
                        'history': hist.history, 'epochs': len(hist.history['loss']),
                        'time': time.time() - t0})
        results.append(metrics)
        print(f"  Fold {fold_idx:2d} | S{test_subj:02d} | Acc={metrics['accuracy']*100:.2f}% | "
              f"Épocas={metrics['epochs']} | {metrics['time']:.0f}s")
    
    accs = [r['accuracy'] for r in results]
    print(f'  → Media: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%')
    return results

print('Funciones listas.')

## 4. Entrenamiento BiLSTM y BiGRU

In [ ]:
# BiLSTM — Todos los joints
results_bilstm_all = train_loso_cv(
    X_data=X, y=y, S=S,
    model_builder_fn=lambda s: build_bilstm(s),
    model_name='BiLSTM (todos joints)'
)

In [ ]:
# BiGRU — Todos los joints
results_bigru_all = train_loso_cv(
    X_data=X, y=y, S=S,
    model_builder_fn=lambda s: build_bigru(s),
    model_name='BiGRU (todos joints)'
)

In [ ]:
# BiGRU — Solo piernas (mejor combinación esperada)
results_bigru_legs = train_loso_cv(
    X_data=X_legs, y=y, S=S,
    model_builder_fn=lambda s: build_bigru(s),
    model_name='BiGRU (solo piernas)'
)

In [ ]:
# BiLSTM — Solo piernas
results_bilstm_legs = train_loso_cv(
    X_data=X_legs, y=y, S=S,
    model_builder_fn=lambda s: build_bilstm(s),
    model_name='BiLSTM (solo piernas)'
)

## 5. Análisis de resultados

In [ ]:
all_bi_results = {
    'BiGRU (todos)':    results_bigru_all,
    'BiGRU (piernas)':  results_bigru_legs,
    'BiLSTM (todos)':   results_bilstm_all,
    'BiLSTM (piernas)': results_bilstm_legs,
}

# Tabla comparativa
print(f'\n{"Modelo":25s} {"Accuracy":>10} {"Sens.":>8} {"Espec.":>8} {"Prec.":>8}')
print('-' * 65)
for name, results in all_bi_results.items():
    accs = [r['accuracy'] for r in results]
    sens = np.mean([r['sensitivity'] for r in results], axis=0).mean()
    spec = np.mean([r['specificity'] for r in results], axis=0).mean()
    prec = np.mean([r['precision'] for r in results], axis=0).mean()
    print(f'{name:25s} {np.mean(accs)*100:9.2f}% {sens*100:7.2f}% {spec*100:7.2f}% {prec*100:7.2f}%')

print('-' * 65)
print('Referencia: GRU-4L (piernas, Jun 2020): 93.67%')
print('Referencia: Bi-LSTM (Guo 2020): 90.75%')

In [ ]:
# Visualización: Accuracy por fold para todos los modelos bidireccionales
n_folds = len(results_bigru_all)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.Set1(np.linspace(0, 1, 4))
fold_labels = [f'S{r["test_subject"]}' for r in results_bigru_all]

# Accuracy por fold
x = np.arange(n_folds)
for i, (name, results) in enumerate(all_bi_results.items()):
    accs = [r['accuracy']*100 for r in results]
    offset = (i - 2) * 0.18
    axes[0].bar(x + offset, accs, 0.18, label=f'{name} ({np.mean(accs):.1f}%)',
                color=colors[i], alpha=0.85, edgecolor='black', linewidth=0.5)

axes[0].set_title('BiLSTM y BiGRU — Accuracy por Fold LOSO-CV', fontweight='bold')
axes[0].set_xlabel('Fold (sujeto de test)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(fold_labels, fontsize=8)
axes[0].set_ylim(40, 115)

# Boxplot de distribución de accuracies
acc_data = [[r['accuracy']*100 for r in results] for results in all_bi_results.values()]
bp = axes[1].boxplot(acc_data, labels=list(all_bi_results.keys()),
                     patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].axhline(93.67, color='green', linestyle='--', alpha=0.8, label='GRU paper (93.67%)')
axes[1].axhline(90.75, color='orange', linestyle='--', alpha=0.8, label='Bi-LSTM Guo (90.75%)')
axes[1].set_title('Distribución de Accuracy (10 folds)', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('bilstm_bigru_results.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de confusión del mejor modelo bidireccional
best_bi_name = max(all_bi_results.keys(),
                   key=lambda k: np.mean([r['accuracy'] for r in all_bi_results[k]]))
best_bi_results = all_bi_results[best_bi_name]
best_bi_acc = np.mean([r['accuracy'] for r in best_bi_results])

print(f'Mejor modelo bidireccional: {best_bi_name} ({best_bi_acc*100:.2f}%)')

cm_total = sum(r['confusion_matrix'] for r in best_bi_results)
cm_pct = cm_total.astype(float) / cm_total.sum(axis=1, keepdims=True) * 100
cls_labels = [CLASS_NAMES[i] for i in range(N_CLASSES)]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=cls_labels, yticklabels=cls_labels,
            ax=ax, cbar_kws={'label': '%'})
ax.set_title(f'Matriz de confusión — {best_bi_name}\nAccuracy={best_bi_acc*100:.2f}%', fontweight='bold')
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('cm_best_bidir.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Métricas por clase del mejor modelo
print(f'\n=== Métricas por clase — {best_bi_name} ===')
print(f'{"Clase":22s} {"Sensibilidad":>14} {"Especificidad":>14} {"Precisión":>12}')
print('-' * 65)
mean_sens = np.mean([r['sensitivity'] for r in best_bi_results], axis=0)
mean_spec = np.mean([r['specificity'] for r in best_bi_results], axis=0)
mean_prec = np.mean([r['precision']   for r in best_bi_results], axis=0)
for cls_id in range(N_CLASSES):
    print(f'{CLASS_NAMES[cls_id]:22s} {mean_sens[cls_id]*100:13.2f}% {mean_spec[cls_id]*100:13.2f}% {mean_prec[cls_id]*100:11.2f}%')
print('-' * 65)
print(f'{"PROMEDIO":22s} {mean_sens.mean()*100:13.2f}% {mean_spec.mean()*100:13.2f}% {mean_prec.mean()*100:11.2f}%')

In [ ]:
# Guardar resultados
results_to_save = {}
for name, results in all_bi_results.items():
    accs = [r['accuracy'] for r in results]
    results_to_save[name] = {
        'accuracies': accs,
        'mean_acc': float(np.mean(accs)),
        'std_acc': float(np.std(accs)),
        'sensitivity': np.mean([r['sensitivity'] for r in results], axis=0).tolist(),
        'specificity': np.mean([r['specificity'] for r in results], axis=0).tolist(),
        'precision':   np.mean([r['precision']   for r in results], axis=0).tolist(),
        'confusion_matrix': sum(r['confusion_matrix'] for r in results).tolist(),
    }

with open('results_bidir.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print('Resultados guardados en results_bidir.pkl')
print('\n→ Siguiente: Notebook 06 - Transformer y Self-Attention')